# Huấn luyện YOLOv8-P2 cho nhận diện biển báo (Zalo AI 2020) trên Google Colab
Notebook này được thiết kế để chạy độc lập trên **Google Colab**. Sử dụng API Kaggle để tải data siêu tốc (không cần Google Drive), tự động cắt ảnh, xử lý JSON sang format YOLO và huấn luyện mô hình YOLOv8-P2.

In [ ]:
# 1. KẾT NỐI GOOGLE DRIVE ĐỂ LƯU KẾT QUẢ TRỌNG SỐ (Chống mất mát khi mất kết nối)
from google.colab import drive
import os
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/DoAn_NhanDienBienBao'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Thư mục sao lưu an toàn: {SAVE_DIR}")


In [ ]:
# 2. TẢI DATASET BẰNG KAGGLE API (SIÊU TỐC)
# Yêu cầu: Bạn phải tải file kaggle.json (từ tài khoản Kaggle của bạn) lên thư mục gốc của Colab trước khi chạy ô này.
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d phhasian0710/za-traffic-2020
!unzip -q -n za-traffic-2020.zip -d /content/dataset


In [ ]:
# 3. CHUẨN BỊ THƯ MỤC VÀ CHIA DATA 80/20
import json
import glob
import shutil
import random
from tqdm import tqdm

json_paths = glob.glob('/content/dataset/**/train_traffic_sign_dataset.json', recursive=True)
if not json_paths:
    raise FileNotFoundError("Không tìm thấy file JSON. Vui lòng kiểm tra lại bước tải dữ liệu!")

json_path = json_paths[0]
image_dir = os.path.dirname(json_path).replace('traffic_train', 'traffic_train/images')

if not os.path.exists(image_dir):
    img_dirs = glob.glob('/content/dataset/**/traffic_train/images', recursive=True)
    if img_dirs:
        image_dir = img_dirs[0]

dataset_dir = '/content/yolo_dataset'
for split in ['train', 'val']:
    os.makedirs(f'{dataset_dir}/{split}/images', exist_ok=True)
    os.makedirs(f'{dataset_dir}/{split}/labels', exist_ok=True)

print("Đang đọc dữ liệu JSON...")
with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

images_info = {img['id']: img for img in data['images']}

img_to_anns = {}
for ann in data['annotations']:
    img_id = ann['image_id']
    if img_id not in img_to_anns:
        img_to_anns[img_id] = []
    img_to_anns[img_id].append(ann)

image_ids = list(images_info.keys())
random.seed(42)
random.shuffle(image_ids)
split_idx = int(len(image_ids) * 0.8)
train_ids = image_ids[:split_idx]
val_ids = image_ids[split_idx:]

print(f"Tổng số ảnh: {len(image_ids)}. Train: {len(train_ids)}, Val: {len(val_ids)}")


In [ ]:
# 4. CHUYỂN ĐỔI COCO SANG YOLO
def convert_coco_to_yolo(bbox, img_width, img_height):
    x_min, y_min, w, h = bbox
    x_center = (x_min + w / 2) / img_width
    y_center = (y_min + h / 2) / img_height
    w_norm = w / img_width
    h_norm = h / img_height
    return x_center, y_center, w_norm, h_norm

print("Đang xử lý và tạo file .txt cho YOLO...")
def process_split(ids, split_name):
    for img_id in tqdm(ids, desc=f"Processing {split_name}"):
        img_info = images_info[img_id]
        img_filename = img_info['file_name']
        img_width = img_info['width']
        img_height = img_info['height']
        
        src_img_path = os.path.join(image_dir, img_filename)
        dst_img_path = os.path.join(dataset_dir, split_name, 'images', img_filename)
        
        if os.path.exists(src_img_path):
            shutil.copy(src_img_path, dst_img_path)
            
            txt_filename = img_filename.rsplit('.', 1)[0] + '.txt'
            txt_path = os.path.join(dataset_dir, split_name, 'labels', txt_filename)
            
            with open(txt_path, 'w') as f_txt:
                if img_id in img_to_anns:
                    for ann in img_to_anns[img_id]:
                        class_id = int(ann['category_id']) - 1
                        x_c, y_c, w_n, h_n = convert_coco_to_yolo(ann['bbox'], img_width, img_height)
                        f_txt.write(f"{class_id} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}\n")

process_split(train_ids, 'train')
process_split(val_ids, 'val')
print("Hoàn tất quá trình chuẩn bị dữ liệu!")


In [ ]:
# 5. TẠO FILE CẤU HÌNH DATASET
yaml_content = f"""
path: {dataset_dir}
train: train/images
val: val/images

names:
  0: No entry
  1: No parking / waiting
  2: No turning
  3: Max Speed
  4: Other prohibition signs
  5: Warning signs
  6: Mandatory signs
"""
with open('/content/dataset.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_content.strip())
print("Đã tạo xong file cấu hình dataset.yaml")


In [ ]:
# 6. TẠO KIẾN TRÚC YOLOv8-P2
!pip install -q ultralytics
p2_yaml_content = """
# Ultralytics YOLO 🚀, AGPL-3.0 license
# YOLOv8-p2 architecture
nc: 7  # number of classes
scales: 
  s: [0.33, 0.50, 1024] 

backbone:
  - [-1, 1, Conv, [64, 3, 2]]  # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]  # 1-P2/4
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]  # 3-P3/8
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]  # 5-P4/16
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]  # 7-P5/32
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]  # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]  # cat backbone P4
  - [-1, 3, C2f, [512]]  # 12

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]  # cat backbone P3
  - [-1, 3, C2f, [256]]  # 15 (P3/8-small)

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 2], 1, Concat, [1]]  # cat backbone P2
  - [-1, 3, C2f, [128]]  # 18 (P2/4-xsmall)

  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]  # cat head P3
  - [-1, 3, C2f, [256]]  # 21 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]  # cat head P4
  - [-1, 3, C2f, [512]]  # 24 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]  # cat head P5
  - [-1, 3, C2f, [1024]]  # 27 (P5/32-large)

  - [[18, 21, 24, 27], 1, Detect, [nc]]  # Detect(P2, P3, P4, P5)
"""
with open('/content/yolov8s-p2.yaml', 'w', encoding='utf-8') as f:
    f.write(p2_yaml_content)
print("Đã tạo kiến trúc mạng YOLOv8s-P2 thành công!")


In [ ]:
# 7. HUẤN LUYỆN MÔ HÌNH
from ultralytics import YOLO

model = YOLO('/content/yolov8s-p2.yaml')
model.load('yolov8s.pt')

results = model.train(
    data='/content/dataset.yaml',
    epochs=50,
    imgsz=1280,
    batch=8,
    max_det=50,
    iou=0.6,
    optimizer='AdamW',
    cos_lr=True,
    cls=2.0,
    box=1.0,
    mosaic=1.0,
    degrees=10.0,
    translate=0.2,
    project='/content/drive/MyDrive/DoAn_NhanDienBienBao/zalo_traffic',
    name='yolov8s_p2_highres',
    device=0,
)


In [ ]:
# 8. THÔNG BÁO HOÀN TẤT
print('🎉 Quá trình huấn luyện đã kết thúc!')
print('Vì chúng ta đã thiết lập project lưu thẳng vào Google Drive, toàn bộ trọng số (weights), biểu đồ Loss, và kết quả đã được cất giữ an toàn tuyệt đối tại: /content/drive/MyDrive/DoAn_NhanDienBienBao/zalo_traffic/yolov8s_p2_highres')
